# Week 16, Day 2: Bayes' rule: priors, likelihoods and posteriors

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w016/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Evidence accumulates

The same stimulus is shown on several independent presentations, and the neuron bursts on each. Using the odds form, compute the posterior probability that it is the preferred stimulus after 0, 1, 2, 3 and 4 bursts, and store them in the array `post_after`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
prior_odds = 0.02 / 0.98
LR = 0.9 / 0.05
post_odds = ...                              # array for k = 0, 1, 2, 3, 4 bursts
post_after = ...                             # convert odds to probabilities: odds / (1 + odds)
print("posterior after k bursts:", post_after.round(4))

In [ ]:
import numpy as np
def test_one():
    "the posterior after one burst, from the odds form, is about 0.269"
    assert abs(post_after[1] - 0.9 * 0.02 / (0.9 * 0.02 + 0.05 * 0.98)) < 1e-9

def test_sequence():
    "after three bursts in three independent presentations the posterior is above 0.98"
    assert post_after[3] > 0.98 and np.all(np.diff(post_after) > 0)

test_one()
test_sequence()
print("All checks passed")

### Exercise: A Bayesian decoder

Write `posterior(counts, prefs, log_prior)` returning the normalised posterior on `grid`. Then, for 4 and 32 neurons, decode 2,000 trials whose orientations are drawn from the prior (centred on 45° with sd 15°), and store the mean squared errors of the ML and MAP estimates in `err[N]['ML']` and `err[N]['MAP']`.

In [ ]:
rng = np.random.default_rng(2002)
grid = np.arange(0, 180, 0.5)
def circ(d): d = np.mod(d, 180); return np.minimum(d, 180 - d)
def tuning(theta, prefs): return 1 + 9 * np.exp(2 * (np.cos(np.deg2rad(2 * (np.subtract.outer(theta, prefs)))) - 1))
log_prior = -circ(grid - 45) ** 2 / (2 * 15 ** 2)
prefs4 = np.arange(4) * 45.0 + 10

def posterior(counts, prefs, log_prior):
    f = tuning(grid, prefs)                                 # shape (len(grid), N)
    loglik = ...
    lp = ...                                                # log posterior up to a constant
    p = np.exp(lp - lp.max())
    return p / p.sum()

err = {}
for N in [4, 32]:
    prefs = np.arange(N) * 180 / N + 10
    sq = {"ML": [], "MAP": []}
    for _ in range(2000):
        theta = np.mod(45 + 15 * rng.normal(), 180)
        counts = rng.poisson(tuning(np.array([theta]), prefs)[0])
        flat = posterior(counts, prefs, np.zeros_like(grid))       # a flat prior gives the likelihood
        post = posterior(counts, prefs, log_prior)
        sq["ML"].append(circ(grid[np.argmax(flat)] - theta) ** 2)
        sq["MAP"].append(...)
    err[N] = {k: np.mean(v) for k, v in sq.items()}
print({N: {k: round(v, 1) for k, v in e.items()} for N, e in err.items()})

In [ ]:
import numpy as np
def test_posterior():
    "posterior returns a normalised distribution over the grid"
    p = posterior(np.array([3, 0, 1, 5]), prefs4, log_prior)
    assert abs(p.sum() - 1) < 1e-9 and np.all(p >= 0)

def test_map_better_few():
    "with 4 neurons, the MAP estimate has a lower mean squared error than ML (stimuli drawn from the prior)"
    assert err[4]["MAP"] < 0.8 * err[4]["ML"]

def test_converge_many():
    "with 32 neurons the two errors are within 15% of each other"
    assert abs(err[32]["MAP"] - err[32]["ML"]) / err[32]["ML"] < 0.15

test_posterior()
test_map_better_few()
test_converge_many()
print("All checks passed")